# EIGSEP Signal Recovery using the New Framework (v001)

**Demonstrates the refactored eigsep_sim architecture:**

- Composed objects: `Observer`, `Beam`, `Sky`, `Terrain`, `ForwardModel`
- Spectral basis decomposition: `BeamBasis`, `SkyBasis`
- Joint optimization: `Calibrator` with Anderson Acceleration + JAX autodiff

**IMPORTANT:** Run cells sequentially from top to bottom!

In [ ]:
import os
import numpy as np
import healpy
import matplotlib.pyplot as plt
from astropy.time import Time
import astropy.units as u

from eigsep_sim import (
    EarthSurface, Beam, Sky, NullTerrain, ForwardModel, Calibrator,
    DTYPE_R_NPY, DTYPE_R_JAX
)
from eigsep_sim.models import T21cmModel
from eigsep_sim.linear_solver import normal_solve
from eigsep_sim.spectral import gsm_eigenmodes, eigenmode_filter

# Configuration
NSIDE = 8
NPIX = healpy.nside2npix(NSIDE)
LAT_DEG, LON_DEG = 39.2, -113.4
FREQS_MHZ = np.linspace(55.0, 150.0, 20)
FREQS_HZ = FREQS_MHZ * 1e6
N_FREQ = len(FREQS_MHZ)
DELTA_NU_HZ = float(np.diff(FREQS_MHZ).mean()) * 1e6
OBS_EPOCH = Time("2025-01-01")
N_TIMES = 24
N_AZ, N_ALT = 4, 3
N_ORIENT = N_AZ * N_ALT
T_RX_K = 100.0

print(f"Config: NSIDE={NSIDE}, {N_FREQ} freqs, {N_TIMES} times, {N_ORIENT} pointings")
print("✓ Ready to proceed")

In [ ]:
# Create Observer, Beam, Sky
obs = EarthSurface(lat=LAT_DEG, lon=LON_DEG)
print(f"✓ Observer created")

beam = Beam.from_dipole(nside=8, freqs_hz=FREQS_HZ, arm_lengths_m=[2.0], K=5)
print(f"✓ Beam created: basis {beam.basis.A.shape}")

sky = Sky.from_gsm(NSIDE, FREQS_HZ, n_modes=5, include_flat=True)
print(f"✓ Sky created: basis {sky.basis.A.shape}")

gsm_maps = sky.init_coeffs()
gsm_maps_recon = gsm_maps @ sky.basis.A.T

fwd = ForwardModel(obs, beam, sky, terrain=NullTerrain())
print(f"✓ ForwardModel created")

In [ ]:
# Precompute geometry
times = OBS_EPOCH + np.linspace(0, 86400, N_TIMES, endpoint=False) * u.s
print(f"Computing geometry for {N_TIMES} times …", flush=True)
geom = fwd.precompute_geometry(times)

masks_all = np.array([geom['masks'][ti] for ti in range(N_TIMES)])
mean_open = masks_all.mean()
print(f"✓ Geometry cached. Mean visibility: {mean_open:.2f}")

In [ ]:
# Forward simulation
sky_coeffs = gsm_maps
beam_coeffs = beam.coeffs.copy()

print(f"Simulating antenna temperature …", flush=True)
antenna_temp = fwd.simulate(sky_coeffs, beam_coeffs, geom=geom)
print(f"✓ Simulation complete: {antenna_temp.shape}")
print(f"  Range: {antenna_temp.min():.1f}–{antenna_temp.max():.1f} K")

In [ ]:
# Create synthetic observations with noise
tau_per_obs = 86400.0 / (N_TIMES * N_ORIENT)
sigma_noise = np.array([
    (gsm_maps_recon[:, fi].mean() + T_RX_K) / np.sqrt(DELTA_NU_HZ * tau_per_obs)
    for fi in range(N_FREQ)
])

antenna_temp_flat = antenna_temp.reshape(-1, N_FREQ)
rng = np.random.default_rng(seed=42)
noise = rng.normal(scale=sigma_noise[None, :], size=antenna_temp_flat.shape)
data_noisy = antenna_temp_flat + noise

print(f"✓ Data created with noise: {data_noisy.shape}")
print(f"  σ_noise: {sigma_noise.mean()*1e3:.1f} mK")

In [ ]:
# Initialize and run Calibrator
cal = Calibrator(
    fwd=fwd,
    data=data_noisy,
    inv_noise_var=1.0 / (sigma_noise[None, :]**2),
    m_anderson=5,
    lam_beam=0.01,
    lam_sky=0.0
)
print(f"✓ Calibrator initialized")

params_init = cal.init_params(times=times)
print(f"Running fit …", flush=True)
result = cal.fit(params=params_init, times=None, max_iter=10, tol=1e-3, verbose=True)
print(f"\n✓ Converged: {result['converged']} in {result['n_iter']} iterations")

In [ ]:
# Plot convergence
fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(result['losses'], 'bo-', markersize=6)
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.set_title('Calibrator Convergence')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

params_opt = result['params']
print(f"\nParameter changes:")
print(f"  Sky RMS change: {np.std(params_opt['sky_coeffs'] - params_init['sky_coeffs']):.6f}")
print(f"  Beam RMS change: {np.std(params_opt['beam_coeffs'] - params_init['beam_coeffs']):.6f}")

## ✅ Success!

This notebook demonstrates the new eigsep_sim framework:

1. **Composed Objects**: Observer, Beam, Sky, Terrain, ForwardModel
2. **Spectral Basis**: BeamBasis & SkyBasis for compact representation
3. **Geometry Caching**: Reuse across parameter sweeps
4. **JAX Simulation**: Autodiff-ready forward model
5. **Calibrator**: Joint optimization with Anderson Acceleration

All components work together seamlessly!